In [24]:
import os 
from dotenv import load_dotenv
from openai import OpenAI
import json 
from bs4 import BeautifulSoup
import requests
from urllib.parse import urljoin
from IPython.display import display,Markdown

In [25]:
load_dotenv(override=True)

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# Function to fetch the content of website
def fetch_website(url) -> str:
    response = requests.get(url, headers=HEADERS, timeout=15)
    response.raise_for_status()
    soup = BeautifulSoup(response.content, "html.parser")
    
    # Handle missing title safely
    title = soup.title.text if soup.title else "No Title"
    
    if soup.body:
        for irrelevantTags in soup.body(["img", "footer", "header", "nav", "script", "form"]):
            irrelevantTags.decompose()
            
    text = soup.get_text(separator="\n", strip=True)
    return title + "\n" + text

In [26]:
# Function to fetch the urls of a website 
def fetch_website_links(url):
    response = requests.get(url, headers=HEADERS, timeout=15)
    response.raise_for_status()
    soup = BeautifulSoup(response.content, "html.parser")
    links = []
    for tag in soup.find_all("a"):
        href = tag.get("href")
        if href:
            links.append(urljoin(url, href))
    return links

In [27]:
system_prompt_message = """
You are provided with the list of links on a webpage. Please suggest which links are useful for generating a brochure for the company.
Respond with full http url with json format.
{
    "links" : [
        {"type" : "About", "url" : "https://full_url_goes_here/"},
        {"type" : "career", "url" : "https://full_url_goes_here/"}
    ]
}
"""


def user_message(url):
    user_prompt = f"""
    Here are the links from a website {url} - I want to generate a brochure for the company. Please suggest to me which links are relevant ones.
    Do not include privacy policy, service policy like things or any policy details or instagram, twitter or any other social media link.
    \n
    """
    links = fetch_website_links(url)
    user_prompt += "\n".join(links) + "\n"
    return user_prompt

In [28]:
openai = OpenAI(
    api_key=os.getenv("OLLAMA_API_KEY"),
    base_url=os.getenv("OLLAMA_HOST")
)

In [29]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=os.getenv("MODEL_NAME"),
        messages=[
            {
                "role": "system",
                "content": system_prompt_message
            },
            {
                "role": "user",
                "content": user_message(url)
            }
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links

In [30]:
def fetch_page_and_all_relevant_links(url):
    content = fetch_website(url)
    relevant_links = select_relevant_links(url)
    result = f"Landing page content {content}\n Relevant Links"
    
    for link in relevant_links['links']:
        # Fixed f-string quote nesting issue using single quotes for 'type' and 'url'
        result += f"\n\n ## link: {link['type']}\n"
        result += fetch_website(link['url'])
        
    return result


# print(fetch_page_and_all_relevant_links(url))

In [ ]:
brosher_system_prompt = """
You are a assistant that analyze the sevral relvant page from company website and create a short brosher about the company ,respetive customer and recurit
respond in markdown without code block - include details of company culture and custiomer and carrer jobs information.
"""

def user_brosher_prompt(company_name, url):
    user_prompt = f"""
    you are loking at company {company_name} and below are the details from all revlant pages form company.
    Make a shot version of brosher for company and respond in markdown with no code block
    """
    user_prompt += fetch_page_and_all_relevant_links(url)
    return user_prompt

# print(user_brosher_prompt("TVF",url))


In [36]:
def create_brochure(company_name,url):
    response = openai.chat.completions.create(
        model= "gemma4", temperature=0.7,
        messages=[{"role":"system","content":brosher_system_prompt},
        {"role":"user","content":user_brosher_prompt(company_name,url)}
        ]
    )

    result = response.choices[0].message.content
    display(Markdown(result))

In [37]:
url = "https://www.theviralfever.com/"
create_brochure("TVF",url)

# 🌟 The Viral Fever (TVF): Change the Narrative 🌟

## 🎬 The Company Story: Lights, Camera, Experiment

Founded in 2012, The Viral Fever (TVF) is not just a content brand—it is a cultural phenomenon. We are one of India's most popular and critically acclaimed media entities, dedicated to transforming the landscape of digital entertainment.

**Our DNA:**
*   **Pioneering Force:** We actively disrupt traditional storytelling, operating at the intersection of youth-centric energy and universal, family-friendly narratives.
*   **Commitment to Quality:** Our content is driven by deep commitment to quality, consistently earning national and international recognition.
*   **Global Acclaim:** TVF originals have been featured among top global shows, securing accolades from prestigious bodies like the Filmfare Awards, International Emmys, and the Asian Academy Creative Awards.

**Our Ecosystem:**
Through our diverse portfolio of original series (like *Panchayat*, *Kota Factory*, *Gullak*, and *Family Kirana Store*), and our vibrant sister channels (*The Timeliners*, *The Screen Patti*, *Girliyapa*), we reach millions, proving that genuine storytelling can resonate across all demographics.

***

## 🎭 Our Audience & Impact: Why We Resonate

At TVF, we don't just create content; we create conversations. Our audience is a massive, engaged community—millions of viewers across India and globally—who crave stories that are true, relatable, and bold.

**Who We Are For:**
*   **The Curious Minds:** Viewers who appreciate nuanced, character-driven narratives that challenge stereotypes.
*   **The Modern Family:** Families who seek entertaining content that balances lighthearted comedy with deep, realistic human experiences.
*   **The Digital Native:** A massive, dynamic audience that consumes content in real-time and demands high quality, authentic digital media.

**Our Promise to Viewers:**
We deliver binge-worthy originals that speak directly to the modern Indian experience—from the complexities of college life to the warmth of small-town life—ensuring every viewer finds a story that feels uniquely *theirs*.

***

## 🚀 Join Our Story: Culture & Career Opportunities

We believe that our greatest stories are told by passionate, brilliant people. If you are ready to move beyond traditional corporate molds and contribute to a dynamic, boundary-pushing creative environment, TVF is where you belong.

**💡 The TVF Culture:**
*   **Creative Freedom:** We foster a culture of intense creativity, experimentation, and fearless storytelling. We encourage team members to challenge assumptions and push creative boundaries.
*   **High Impact:** You are not a cog in a machine; you are a storyteller. Your work directly impacts millions of lives, making every day meaningful.
*   **Collaborative Energy:** Our environment is fueled by diverse talent—writers, directors, technicians, and strategists—all united by a shared passion for making culturally impactful media.

**💼 Careers at TVF:**
We are always looking for individuals who are passionate, dedicated, and ready to disrupt the status quo. Our roles are diverse and span the entire content lifecycle:

*   **Creative Roles:** Writers, Screenwriters, Directors, Showrunners, and Narrative Consultants.
*   **Production & Technical:** Cinematographers, Editors, Post-Production Specialists, and Sound Engineers.
*   **Digital & Business:** Digital Marketing Strategists, Community Managers, and Brand Partners.

**Ready to be part of the narrative?**
If you are driven by the power of a good story and want to work with a brand that is constantly setting global standards in Indian media, we invite you to join The Viral Fever family.

In [ ]:
# streaming response
from IPython.display import update_display


def stream_brochure(company_name,url):
    stream = openai.chat.completions.create(
        model= "gemma4", temperature=0.7,
        messages=[
            {"role":"system","content":brosher_system_prompt},
            {"role":"user","content":user_brosher_prompt(company_name,url)}
        ],
        stream=True
    )

    response = ""
    # 2. Added display_id=True so the output can be updated live
    display_handle = display(Markdown(" "),display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or " "
        display_handle.update(Markdown(response))

url = "https://www.theviralfever.com/"
stream_brochure("TVF",url)


# 🎬 The Viral Fever (TVF): Where Stories Come to Life

## ✨ Our Story: The Pioneers of Indian Digital Entertainment

Since 2012, The Viral Fever (TVF) has been more than just a media brand—we are a cultural phenomenon. Founded by Arunabh Kumar, we set out to prove that Indian storytelling could be both deeply rooted and globally resonant.

We began as a group of IIT engineers, driven by a simple philosophy: **Lights · Camera · Experiment.**

Today, we are one of India's most critically acclaimed content powerhouses, consistently delivering original series that not only entertain but also challenge stereotypes and connect with millions. Our success is marked by national and international accolades, including recognition from the Filmfare Awards, the International Emmys, and the Asian Academy Creative Awards.

**🚀 Our Scale:**
*   **Content Universe:** TVF, The Timeliners, Girliyapa, and The Screen Patti.
*   **Digital Reach:** Millions of subscribers across our sister channels.
*   **Impact:** From web series like *Panchayat* and *Kota Factory* to cinematic releases like *The Vvaan*, we operate across digital and theatrical platforms.

***

## 🌟 For Our Partners & Audiences: The Power of Storytelling

At TVF, we believe that the most effective content doesn't just *contain* a brand—it *is* the brand's narrative. We specialize in crafting experiences that are organic, relatable, and unforgettable.

**💡 What We Offer:**

*   **Binge-Worthy Originals:** High-quality, youth-centric, and family-friendly web series that consistently dominate conversations (e.g., *Gullak*, *The Family Kirana Store*).
*   **Native Branded Content:** We move beyond simple product placement. Our collaborations (with brands like Asian Paints, Oral-B, and Birla Tyres) integrate products seamlessly into existing, beloved universes, ensuring the message feels native, not inserted.
*   **Cultural Resonance:** We excel at taking real consumer insights and weaving them into compelling characters and relatable everyday situations, turning functional benefits into entertaining narratives.
*   **Full Spectrum Reach:** Whether it’s a short micro-drama or a large-scale pan-Indian film, we deliver stories that transcend platforms.

***

## 💖 Our Culture: The TVF Ethos

Our culture is defined by a commitment to authentic creativity. We are a dynamic, forward-thinking collective who treats storytelling as an experiment.

**We are driven by:**
*   **Creativity & Experimentation:** We constantly push boundaries, asking "what if?" to create unique, high-impact content.
*   **Cultural Authenticity:** Our stories are always rooted in the nuances of everyday India, giving our content a genuine, familiar, and emotional core.
*   **Quality Over Convenience:** We maintain a fierce commitment to storytelling quality, ensuring our content is not just viral, but critically acclaimed.
*   **A Place for Ideas:** We foster an environment where ideas—whether a script, a business concept, or a creative solution—are welcomed and nurtured.

***

## 🤝 Join Our Journey: Careers & Collaboration

Are you ready to contribute to a content machine that truly makes a difference? Whether you are an aspiring creator or a seasoned industry professional, we invite you to be part of our wonderful journey.

**💼 Career Opportunities (Recruitment):**
As a growing network, we are always looking for passionate talent across various fields:
*   Writers and Scriptwriters
*   Video Editors and Producers
*   Content Strategists
*   Marketing and Social Media Managers

**✉️ Get Involved:**
*   **Join the Team:** Share your CV and portfolio to become part of the TVF family.
*   **Pitch Your Idea:** Have a compelling story? Send us your script!
*   **Business Enquiries:** Partner with us to create impactful, culturally relevant narratives for your brand.

**Contact Us Today:**
*   **Careers:** careers@theviralfever.com
*   **Scripts:** scripts@theviralfever.com
*   **Partnerships:** business@theviralfever.com